# Transformer Dataset Prep and Validation Split - Christian De Guzman

This notebook turns the labeled contract chunks from `Labeling.ipynb` into a dataset the RoBERTa model can train on (Task #2). It:
1. loads the labeled chunks (`data/processed/*_chunks_labeled.parquet`) and the 41 categories,
2. moves about 15% of the **training contracts** into a validation set, so the test set is only used for final results,
3. builds the train / validation / test splits, with each chunk's 41 labels as a list of floats,
4. tokenizes each chunk with the `roberta-base` tokenizer (`input_ids` and `attention_mask`),
5. checks the splits, labels and tokens,
6. saves everything as a Hugging Face `DatasetDict` in `data/processed/roberta_dataset/`,
7. summarizes positives per category in each split.

**Why a validation set?** Training choices (epochs, class weights, decision thresholds) need to be checked on data the model hasn't trained on. If we used the test set for that, the final test scores would look better than they really are. The split is **by contract**, because neighboring chunks overlap by 128 tokens: splitting by chunk would put almost-identical text in both train and validation.

## 1. Setup

In [1]:
import json, os, shutil, tempfile
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Sequence, Value, load_from_disk
from transformers import AutoTokenizer

C:\Users\CJ\accenture_1h\Accenture-1H-contract-review-challenge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


All settings live in one place. Paths are relative to the `notebooks/` folder, like the other notebooks.

In [2]:
SEED = 42               # starting seed for the validation split
VAL_FRACTION = 0.15     # share of training contracts moved to validation
MAX_SPLIT_TRIES = 1000  # seeds to try before giving up on category coverage
MAX_LENGTH = 512        # roberta-base input limit
TOKENIZER_NAME = "roberta-base"

DATA_DIR = "../data/processed"
VAL_TITLES_PATH = f"{DATA_DIR}/val_titles.json"
DATASET_PATH = f"{DATA_DIR}/roberta_dataset"

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"{DATA_DIR} not found. Run this notebook from the notebooks/ folder.")

## 2. Load the labeled chunks
These are the files saved by `Labeling.ipynb`. Each row is one chunk (up to 512 RoBERTa tokens, 128-token overlap) with `title`, `chunk_id`, `char_start`, `char_end`, `chunk_text`, then one 0/1 column for each of the 41 categories, in `categories.json` order.

In [3]:
train_df = pd.read_parquet(f"{DATA_DIR}/train_chunks_labeled.parquet")
test_df = pd.read_parquet(f"{DATA_DIR}/test_chunks_labeled.parquet")
with open(f"{DATA_DIR}/categories.json", "r") as file:
    CATEGORIES = json.load(file)

print("Train chunks:", len(train_df), "| Test chunks:", len(test_df), "| Categories:", len(CATEGORIES))

Train chunks: 14181 | Test chunks: 2898 | Categories: 41


In [4]:
assert len(CATEGORIES) == 41
assert list(train_df.columns[5:]) == CATEGORIES and list(test_df.columns[5:]) == CATEGORIES
assert train_df.shape == (14181, 46) and test_df.shape == (2898, 46)
assert train_df["title"].nunique() == 408 and test_df["title"].nunique() == 102
print("Load checks passed.")

Load checks passed.


## 3. Validation split
About 15% of the 408 training contracts (61) move to a validation set. Whole contracts move, never single chunks.

**Why not just a random 15%?** Some categories are in very few contracts (Source Code Escrow: 12, Unlimited/All-You-Can-Eat-License: 14, Price Restrictions: 15). A random pick of 61 contracts can easily leave one of them with no validation examples, and then its threshold and class weight can't be tuned in Task #3. So the split is random, but **checked**: starting from seed 42, a seed is only accepted if every category has at least one contract on both the train side and the validation side. If a seed fails, the next one is tried.

**The split never changes once made.** The chosen titles are saved to `data/processed/val_titles.json` (committed to git). When that file exists, this notebook loads it instead of splitting again, so everyone on the team uses the same validation set.

In [5]:
def contract_presence(df, categories):
    # One row per contract: True if any of its chunks is labeled with that category
    return df.groupby("title")[categories].max().astype(bool)

In [6]:
def make_val_split(presence, val_fraction=VAL_FRACTION, seed=SEED, max_tries=MAX_SPLIT_TRIES):
    # Returns (sorted validation titles, seed used). A seed is accepted only if every
    # category has at least one contract on both the train side and the validation side.
    n_val = round(val_fraction * len(presence))
    impossible = [c for c in presence.columns if presence[c].sum() < 2]
    if impossible:
        raise ValueError(f"Categories in fewer than 2 contracts can't be on both sides: {impossible}")

    titles = np.array(sorted(presence.index))
    for s in range(seed, seed + max_tries):
        val = sorted(np.random.default_rng(s).permutation(titles)[:n_val])
        val_side, train_side = presence.loc[val].any(), presence.drop(val).any()
        if val_side.all() and train_side.all():
            return val, s
    missing = sorted(set(presence.columns[~val_side]) | set(presence.columns[~train_side]))
    raise ValueError(f"No seed in {max_tries} tries covered every category. Last attempt missed: {missing}")

In [7]:
def load_or_create_split(path, presence):
    # Reuse the saved split if it exists (after checking it), otherwise make one and save it
    if os.path.exists(path):
        with open(path, "r") as file:
            info = json.load(file)
        unknown = [t for t in info["val_titles"] if t not in presence.index]
        if unknown:
            raise ValueError(f"{path} has titles that aren't training contracts: {unknown}")
        if len(info["val_titles"]) != info["n_val_contracts"]:
            raise ValueError(f"{path}: expected {info['n_val_contracts']} titles, found {len(info['val_titles'])}")
        return info

    val, used_seed = make_val_split(presence)
    info = {"seed": used_seed, "val_fraction": VAL_FRACTION, "n_val_contracts": len(val), "val_titles": val}
    with open(path, "w") as file:
        json.dump(info, file, indent=1)
    return info

In [8]:
presence = contract_presence(train_df, CATEGORIES)
split_info = load_or_create_split(VAL_TITLES_PATH, presence)
val_titles = set(split_info["val_titles"])
train_titles = set(presence.index) - val_titles

print("Seed used:", split_info["seed"])
print("Train contracts:", len(train_titles), "| Validation contracts:", len(val_titles))

Seed used: 42
Train contracts: 347 | Validation contracts: 61


### Testing the split function on a toy example
20 made-up contracts and 2 categories, plus a category `C` that is in only one contract (so it can never be on both sides).

In [9]:
toy = pd.DataFrame({"A": [True]*10 + [False]*10, "B": [i % 4 == 0 for i in range(20)]},
                   index=[f"c{i}" for i in range(20)])
v1, s1 = make_val_split(toy, val_fraction=0.25, seed=42)
v2, s2 = make_val_split(toy, val_fraction=0.25, seed=42)
assert (v1, s1) == (v2, s2)                                     # same seed, same split
assert len(v1) == 5 and s1 >= 42
assert toy.loc[v1].any().all() and toy.drop(v1).any().all()     # both sides cover every category
rare = toy.assign(C=[True] + [False]*19)                       # C is in only 1 contract
try:
    make_val_split(rare, val_fraction=0.25, seed=42)
    raise AssertionError("should have raised")
except ValueError as e:
    assert "C" in str(e)                                        # the error names the category
print("Split function toy tests passed.")

Split function toy tests passed.


### Checking the real split
- 61 validation contracts, no contract on both sides, all 408 training contracts kept, no overlap with test.
- Every category has at least one contract in train and in validation.
- Loading the saved file again gives the same split.
- A `val_titles.json` with a title that isn't a training contract is rejected (protects against a stale or hand-edited file).

In [10]:
assert split_info["n_val_contracts"] == 61 == len(val_titles)
assert val_titles <= set(presence.index) and not (val_titles & train_titles)
assert val_titles | train_titles == set(train_df["title"])                       # all 408 kept
assert not (val_titles | train_titles) & set(test_df["title"])
assert presence.loc[sorted(val_titles)].any().all() and presence.loc[sorted(train_titles)].any().all()
assert set(load_or_create_split(VAL_TITLES_PATH, presence)["val_titles"]) == val_titles   # reproducible

bad = os.path.join(tempfile.mkdtemp(), "bad.json")
with open(bad, "w") as file:
    json.dump({**split_info, "val_titles": split_info["val_titles"][:-1] + ["NOT A REAL CONTRACT"]}, file)
try:
    load_or_create_split(bad, presence)
    raise AssertionError("should have raised")
except ValueError as e:
    assert "NOT A REAL CONTRACT" in str(e)
print("Split checks passed.")

Split checks passed.


## 4. Build train / validation / test
- **train:** chunks from the 347 training contracts not in validation
- **validation:** chunks from the 61 validation contracts
- **test:** all test chunks, unchanged

Each chunk's 41 label columns become one `labels` list of `float32` values (0.0 / 1.0), in `categories.json` order. The training loss (`BCEWithLogitsLoss`) expects float labels. The column types are set explicitly so Arrow can't quietly turn them into `float64`.

In [11]:
FEATURES = Features({
    "title": Value("string"),
    "chunk_id": Value("int64"),
    "chunk_text": Value("string"),
    "labels": Sequence(Value("float32"), length=len(CATEGORIES)),
})

def build_split(df, titles, categories):
    # Keep the chunks of these contracts, with the 41 label columns combined into one float32 list
    part = df[df["title"].isin(titles)].reset_index(drop=True)
    out = part[["title", "chunk_id", "chunk_text"]].copy()
    out["labels"] = list(part[categories].to_numpy(dtype=np.float32))
    return Dataset.from_pandas(out, features=FEATURES, preserve_index=False)

In [12]:
raw = DatasetDict({
    "train": build_split(train_df, train_titles, CATEGORIES),
    "validation": build_split(train_df, val_titles, CATEGORIES),
    "test": build_split(test_df, set(test_df["title"]), CATEGORIES),
})
raw

DatasetDict({
    train: Dataset({
        features: ['title', 'chunk_id', 'chunk_text', 'labels'],
        num_rows: 12112
    })
    validation: Dataset({
        features: ['title', 'chunk_id', 'chunk_text', 'labels'],
        num_rows: 2069
    })
    test: Dataset({
        features: ['title', 'chunk_id', 'chunk_text', 'labels'],
        num_rows: 2898
    })
})

In [13]:
assert raw["train"].num_rows + raw["validation"].num_rows == 14181 and raw["test"].num_rows == 2898
for name, ds in raw.items():
    assert ds.features["labels"].feature.dtype == "float32"
    y = np.array(ds["labels"], dtype=np.float32)
    assert y.shape[1] == 41 and set(np.unique(y)) <= {0.0, 1.0}
tv = np.array(raw["train"]["labels"]).sum(0) + np.array(raw["validation"]["labels"]).sum(0)
assert (tv == train_df[CATEGORIES].sum().to_numpy()).all()                 # no labels lost or reordered
assert (np.array(raw["test"]["labels"]).sum(0) == test_df[CATEGORIES].sum().to_numpy()).all()
print("Split build checks passed.")

Split build checks passed.


## 5. Tokenize
The `roberta-base` tokenizer turns each chunk's text into `input_ids` (positions in RoBERTa's vocabulary) and an `attention_mask` (1 for every real token). It's the same tokenizer the chunks were cut with in `Subword_Tokenization_Chunking.ipynb`, so every chunk fits in 512 tokens.

**No padding here.** Chunks keep their real length. Task #2 pads each batch at training time with `DataCollatorWithPadding`, which is faster and takes less space than padding every chunk to 512.

In [14]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)

def tokenize_split(ds, tokenizer, max_length=MAX_LENGTH):
    # Adds input_ids and attention_mask; the raw text is no longer needed after this
    return ds.map(lambda batch: tokenizer(batch["chunk_text"], max_length=max_length, truncation=True),
                  batched=True, remove_columns=["chunk_text"])

In [15]:
dataset = DatasetDict({name: tokenize_split(ds, tokenizer) for name, ds in raw.items()})
dataset

Map:   0%|          | 0/12112 [00:00<?, ? examples/s]

Map:   8%|▊         | 1000/12112 [00:00<00:03, 2890.50 examples/s]

Map:  17%|█▋        | 2000/12112 [00:00<00:02, 3862.15 examples/s]

Map:  25%|██▍       | 3000/12112 [00:00<00:02, 4407.38 examples/s]

Map:  33%|███▎      | 4000/12112 [00:00<00:01, 4499.39 examples/s]

...

Map:  35%|███▍      | 1000/2898 [00:00<00:00, 2975.55 examples/s]

Map:  69%|██████▉   | 2000/2898 [00:00<00:00, 3946.25 examples/s]

Map: 100%|██████████| 2898/2898 [00:00<00:00, 4193.85 examples/s]

Map: 100%|██████████| 2898/2898 [00:00<00:00, 3937.83 examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'chunk_id', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 12112
    })
    validation: Dataset({
        features: ['title', 'chunk_id', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2069
    })
    test: Dataset({
        features: ['title', 'chunk_id', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2898
    })
})

### Checking the tokens
- Every chunk has at most 512 tokens and starts and ends with RoBERTa's special tokens (`<s>` and `</s>`).
- Most chunks are exactly 512 tokens, because the chunking notebook filled each window to the limit. That's expected.
- **About 11% of chunks lose 1 to 3 tokens at the end.** When a chunk is cut out of the middle of a contract, its first word loses the space in front of it, and RoBERTa splits a word without a leading space into slightly more pieces. That pushes a full window 1–3 tokens past 512, so truncation drops the last 1–3 tokens. **No text is actually lost:** the end of every chunk is inside the 128-token overlap, so the next chunk contains those tokens. The checks below confirm that at most a few tokens are dropped (far less than the 128-token overlap), and that no contract's **last** chunk, which has no next chunk to cover it, is truncated.
- Decoding chunks that weren't truncated gives back the original `chunk_text`, which confirms the right tokenizer was used.

In [16]:
for name, ds in dataset.items():
    assert ds.column_names == ["title", "chunk_id", "labels", "input_ids", "attention_mask"]
    for ids in ds["input_ids"]:
        assert len(ids) <= MAX_LENGTH and ids[0] == tokenizer.bos_token_id and ids[-1] == tokenizer.eos_token_id

OVERLAP = 128   # stride used in Subword_Tokenization_Chunking.ipynb
full_lengths = {}
for name, ds in raw.items():
    lengths = np.array([len(ids) for ids in tokenizer(list(ds["chunk_text"]))["input_ids"]])
    full_lengths[name] = lengths
    dropped = np.maximum(lengths - MAX_LENGTH, 0)
    chunks = ds.select_columns(["title", "chunk_id"]).to_pandas()
    is_last = (chunks["chunk_id"] == chunks.groupby("title")["chunk_id"].transform("max")).to_numpy()
    assert dropped.max() < OVERLAP, f"{name}: a chunk lost {dropped.max()} tokens, more than the overlap covers"
    assert (dropped[is_last] == 0).all(), f"{name}: a contract's last chunk was truncated"
    print(f"{name}: {(dropped > 0).sum():,} of {len(ds):,} chunks lose {dropped[dropped > 0].min() if (dropped > 0).any() else 0}-{dropped.max()} "
          f"tokens at the end (covered by the next chunk's overlap); last chunks truncated: 0")

src = raw["validation"]
not_cut = [i for i, n in enumerate(full_lengths["validation"]) if n <= MAX_LENGTH][:3]
for i in not_cut:
    decoded = tokenizer.decode(dataset["validation"][i]["input_ids"], skip_special_tokens=True)
    assert decoded == src[i]["chunk_text"] or " ".join(decoded.split()) == " ".join(src[i]["chunk_text"].split())
print("Tokenization checks passed.")

Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


train: 1,339 of 12,112 chunks lose 1-3 tokens at the end (covered by the next chunk's overlap); last chunks truncated: 0


validation: 260 of 2,069 chunks lose 1-3 tokens at the end (covered by the next chunk's overlap); last chunks truncated: 0


test: 379 of 2,898 chunks lose 1-3 tokens at the end (covered by the next chunk's overlap); last chunks truncated: 0
Tokenization checks passed.


## 6. Save and reload
The dataset is saved to `data/processed/roberta_dataset/`. That folder is **not in git** (it's large, and anyone can rebuild it identically by running this notebook, since `val_titles.json` is committed). Any old copy is deleted first, so re-running the notebook always gives a fresh save.

In [17]:
def save_dataset(dsd, path):
    # Replace any earlier save so a re-run never mixes old and new files
    if os.path.exists(path):
        shutil.rmtree(path)
    dsd.save_to_disk(path)

save_dataset(dataset, DATASET_PATH)
size_mb = sum(os.path.getsize(os.path.join(root, f)) for root, _, files in os.walk(DATASET_PATH) for f in files) / 1e6
print(f"Saved to {DATASET_PATH} ({size_mb:.1f} MB)")

Saving the dataset (0/1 shards):   0%|          | 0/12112 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 12112/12112 [00:00<00:00, 531534.50 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 12112/12112 [00:00<00:00, 514194.72 examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2069 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 2069/2069 [00:00<00:00, 273642.20 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 2069/2069 [00:00<00:00, 254055.13 examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2898 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 2898/2898 [00:00<00:00, 356015.85 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 2898/2898 [00:00<00:00, 328524.91 examples/s]

Saved to ../data/processed/roberta_dataset (47.2 MB)


In [18]:
reloaded = load_from_disk(DATASET_PATH)
for k in ["train", "validation", "test"]:
    assert reloaded[k].num_rows == dataset[k].num_rows and reloaded[k].column_names == dataset[k].column_names
    assert reloaded[k][0] == dataset[k][0]
    assert reloaded[k].features["labels"].feature.dtype == "float32"
del reloaded   # release the saved files so a re-run can replace the folder (Windows locks open files)
print("Save/reload checks passed.")

Save/reload checks passed.


## 7. Summary

### Positive chunks per category in each split
This is how many "yes" examples the model gets for each category. Rare categories have very few, especially in validation and test, so their scores will be noisy.

In [19]:
positives = pd.DataFrame(
    {name: np.array(dataset[name]["labels"]).sum(0).astype(int) for name in ["train", "validation", "test"]},
    index=CATEGORIES,
).sort_values("train")

print("Zero test positives:", list(positives.index[positives["test"] == 0]))
print("Under 5 validation positive chunks:", list(positives.index[positives["validation"] < 5]))
positives

Zero test positives: ['Price Restrictions']
Under 5 validation positive chunks: ['Most Favored Nation', 'Affiliate License-Licensor']


,train,validation,test
Price Restrictions,26,8,0
Unlimited/All-You-Can-Eat-License,26,5,5
Third Party Beneficiary,33,9,15
Source Code Escrow,38,9,2
No-Solicit Of Customers,45,8,18
Most Favored Nation,47,4,5
Non-Disparagement,56,8,15
Affiliate License-Licensor,60,3,22
No-Solicit Of Employees,73,13,19
Joint Ip Ownership,79,20,15


### Results
- **Split:** 408 training contracts became 347 train + 61 validation (seed 42; every category has at least one contract on both sides). The 102 test contracts are unchanged.
- **Chunks:** see the `dataset` output in section 5 for the exact train / validation / test chunk counts (train + validation = 14,181; test = 2,898).
- **Columns in every split:** `title`, `chunk_id`, `input_ids`, `attention_mask`, `labels` (41 `float32` values in `categories.json` order).
- **Tokens:** every chunk fits in 512 tokens. About 11% of chunks lose 1–3 tokens at the end, which the next chunk's 128-token overlap covers (see section 5).

### Files
| File | In git | What it is |
|---|---|---|
| `data/processed/val_titles.json` | Yes | The 61 validation contracts and the seed used. Reused on every run, so the split never changes. |
| `data/processed/roberta_dataset/` | No | The tokenized `DatasetDict`. Rebuild it by running this notebook (about 1–2 minutes). |

### Limitations
- **Price Restrictions has 0 positive test chunks** (see the list above), so its test F1 can't be computed. The team needs to decide whether to report it as N/A or merge or drop it.
- Rare categories have only a handful of validation positives, so tuning thresholds or weights for them will be noisy.
- Labels are inherited from `Labeling.ipynb`: a chunk is positive if any part of a clause overlaps it, even a small piece.

### How to load (Task #2)
```python
from datasets import load_from_disk
from transformers import AutoTokenizer, DataCollatorWithPadding

ds = load_from_disk("../data/processed/roberta_dataset")   # run this notebook first
train_ds, val_ds, test_ds = ds["train"], ds["validation"], ds["test"]

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
collator = DataCollatorWithPadding(tokenizer)               # pads each batch at training time
# Drop "title" and "chunk_id" before batching for the model; keep them for contract-level evaluation.
```